# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [37]:
# import Bio.SeqIO
# import Bio.Seq

# # SeqIO.parse returns reads one at a time; list() pulls them all into memory
# seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

# print(f"Found {len(seqreads)} sequencing reads.")

# read = seqreads[0]

# print(type(read))
print(read.seq.reverse_complement())

TCTGCTATTCCGTGTCAATAAACATCGTCGTTGCATGCCTTCATTTTAGTGCTGCGTTTACAAAACTGGTTTCAAAATTATTAATTTCAGCTTTTTTTTTTTTTTCAAGCAGAAGACGGCATACGAGATTCTGACATGTGACTGGAGTTCAGACGTGTGCTCTTCCGATCTTGGCCTGATGGAGCAGAACTCCCTTTCACGATAGATAAATAATAGTGCACCATAGGCGGCCGCACTAAATAACTTAAGC


In [43]:
print(seqreads[888].seq.reverse_complement()[-53:])

CACGATAGATAAATAATAGTGCACCATAGGCGGCCGCACCACCGCCACACCCT


In [ ]:

import re
import Bio.SeqIO
import Bio.Seq

## test_cases 

test_cases = [
    # (read, expected result, label saying which requirement this checks)
    ("CACGATAGATAAATAATAGTGCACCATAGGCGGCCGCACCACCGCCACACCCT", "AAAAAAAAAAAAAAAA", "positive: clean read"),
    ("TTTTTTTACGTTTTTTGCGGGCCT", None, "requirement 1: upstream sequence is wrong"),
    ("TTTTTACGTTTTTTGCGGCCGCCT", None, "requirement 2: barcode is not 16 base pairs long"),
    ("TTTTYYTACGTTTTTTGCGGCCGCCT", None, "requirement 3: barcode base includes characters other than A, C, T, G, N"),
    ("TTTTTTTtTTTTTTTTGCGGCCGCCT", "AAAAAAAAAAAAAAAA", "barcode contains lowercase but satisfactory"),
    ("AGGCGGCCGCAAAAAAAAAAAAAAAA", None, "requirement 4: input read should be in the reverse orientation")
]

def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"


def classify_HA_NA(seqread, bclen=16, upstream_NA = "CACGATAGATAAATAATAGTGCACCAT", upstream_HA="CCGGATTTGCATATAATGATGCACCAT", middle_section="AGGCGGCCGC"):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream_NA: Sequence expected immediately upstream of the barcode in influenza virus NA gene (forward orientation).
        upstream_HA: Sequence expected immediately upstream of the barcode in influenza virus HA gene (forward orientation).

    Returns:
        "HA" or "NA" or
        None if the read does not match the expected design
        None if the read does not match the expected design.

    Example:
        >>> read_barcode(Bio.Seq.Seq("TTTTTTTTTTTTTTTTGCGGCCGCCT"), bclen=16)
        'HA'
    """

    seqread_upper=str(seqread).upper()
    seqread_original=seqread_upper = str(Bio.Seq.Seq(seqread_upper).reverse_complement())
    HA_pattern=re.compile(f"(?P<HA>{upstream_HA}){middle_section}(?P<barcode>[ACGTN]{{{bclen}}})")
    NA_pattern=re.compile(f"(?P<NA>{upstream_NA}){middle_section}(?P<barcode>[ACGTN]{{{bclen}}})")
    none_pattern=re.compile(f"{middle_section}(?P<barcode>[ACGTN]{{{bclen}}})")
    barcode_misformatted_HA=re.compile(f"{middle_section}(?P<barcode>[A-Z]{{{bclen}}})")

    match_HA = HA_pattern.search(seqread_original)
    match_NA = NA_pattern.search(seqread_original)
    match_none=none_pattern.search(seqread_original)
    match_misformatted_NA=barcode_misformatted.search(seqread_original)
    match_misformatted_HA=barcode_misformatted.search(seqread_original)
    
        
    if match_HA:
        dict_input=["HA", match_HA.group("barcode")]
    elif match_NA: 
        dict_input=["NA", match_NA.group("barcode")]
    elif match_none:
        dict_input=["non_HA_non_NA", match_none.group("barcode")]
    elif:
        dict_input=["misformatted", match_misformatted.group("barcode")]


count_NA=0
count_HA=0
count_nonclassified=0
count_
barcodes=[]
i=0
for seqread in seqreads: 
    gene = classify_HA_NA(seqread.seq)
    if gene == "HA":
        count_HA+=1
        barcodes[i]=barcode
    elif gene == "NA":
        count_NA+=1
    else: 
        count_nonclassified+=1
    i+=1

print(f"There are {count_NA} NA genes")
print(f"There are {count_HA} HA genes")
print(f"There are {count_nonclassified} nonclassified genes")



There are 3912 NA genes
There are 5250 HA genes
There are 838 nonclassified genes


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [ ]:
# your code here...

3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [ ]:
# your code here...